# Running a DAS document, and watching the plan fill in

One document, start to finish: JSON on disk → a validated spec → a plan → one nnsight
session → results sitting on the plan → files.

The document is [`documents/v2/das.json`](../documents/v2/das.json), written in
causalab-mini's own plan-shaped format. It fits a rank-8 rotation at layer 0 of a tiny
CPU Llama, scores the fitted rotation, and saves it.

Everything here runs on CPU in about ten seconds.

## 1. The document

In [1]:
import json, pathlib, pickle, torch

REPO = pathlib.Path.cwd().parent
raw = json.loads((REPO / 'documents' / 'v2' / 'das.json').read_text())

print('root keys:', list(raw))
print('steps    :', list(raw['steps']))

root keys: ['header', 'model', 'roles', 'sites', 'featurizers', 'interventions', 'steps']
steps    : ['fit', 'score', 'weights']


`Spec` is a pydantic model, so parsing *is* validating. An unknown key anywhere, a site
that isn't declared, a save naming something its step doesn't produce — each is refused
here, with the path to it, before anything is loaded.

In [2]:
from causalab_mini.plan.spec import Spec

spec = Spec.model_validate(raw)
spec.steps['fit'].early_stop

EarlyStop(metric='iia', mode='max', patience=3)

In [3]:
# what a refusal looks like
from pydantic import ValidationError

broken = json.loads(json.dumps(raw))
broken['interventions']['das']['reads']['v_cf']['shuffle'] = {'seed': 1}
try:
    Spec.model_validate(broken)
except ValidationError as refusal:
    print(refusal)

1 validation error for Spec
interventions.das.reads.v_cf.shuffle
  Extra inputs are not permitted [type=extra_forbidden, input_value={'seed': 1}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden


## 2. The engine loads the model

An engine *is* a runtime: it loads the model and it knows how to reach inside it. Here
that's nnterp's `StandardizedTransformer` under nnsight.

In [4]:
from causalab_mini.engine import NNterpEngine

engine = NNterpEngine.load(spec.model, device_map='cpu')
print(type(engine.model).__name__, '|', engine.num_layers, 'layers |', engine.width(engine.locate('block_output', 0)), 'wide')

[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got -1. This may result in unexpected behavior.


INFO | nnterp | Auto-detected AutoModelForCausalLM for hf-internal-testing/tiny-random-LlamaForCausalLM (config: LlamaConfig)


[transformers] The following generation flags are not valid and may be ignored: ['pad_token_id']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


StandardizedTransformer | 2 layers | 16 wide


/home/localjadenfk/wd/causalab-mini/.venv/lib/python3.13/site-packages/torch/cuda/__init__.py:228: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12050). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


## 3. Compiling: document → plan

This is the only place the model is consulted on the client. It resolves what the block
may not decide for itself — the addresses, the widths, the tokenizer, which rows, which
absolute positions.

In [5]:
from causalab_mini import plan as plan_module

plan = plan_module.build_spec(spec, REPO / 'documents' / 'data', engine)
list(plan.steps)

['featurizers', 'fit', 'score', 'weights']

### The tree

`featurizers` is the one step the compiler adds — declaring a parameter set is what
builds it. The rest are the document's own steps, in order.

In [6]:
from causalab_mini.plan import Featurizers, Fit, Observe, Plan, Weights


def show(step, name='<root>', indent=0):
    pad = '  ' * indent
    saves = [s.file_path for s in step.saves]
    tail = f'  saves={saves}' if saves else ''
    if isinstance(step, Plan):
        print(f'{pad}{name}: Plan{tail}')
        for child, one in step.steps.items():
            show(one, child, indent + 1)
    elif isinstance(step, Featurizers):
        print(f'{pad}{name}: Featurizers{tail}')
        for one in step.specs:
            print(f'{pad}    {one.name}: {one.kind} k={one.k} d={one.d} seed={one.seed} trained={one.trained}')
    elif isinstance(step, Fit):
        print(f'{pad}{name}: Fit  {len(step.epochs)} epochs x {len(step.epochs[0])} update  '
              f'lr={step.lr}  objective={step.objective}{tail}')
        show(step.epochs[0][0], 'epochs[0][0]', indent + 2)
        show(step.evaluation, 'evaluation', indent + 2)
    elif isinstance(step, Observe):
        print(f'{pad}{name}: Observe  metrics={[m.name for m in step.metrics]}{tail}')
        for f in step.forwards:
            print(f'{pad}    forward {f.name!r} on {f.input!r}  ({len(f.input_ids)} rows x {len(f.input_ids[0])} tokens)')
            for tap in f.taps:
                a = tap.address
                where = a.component + (f'[{a.layer}]' if a.layer is not None else '')
                for w in tap.writes:
                    print(f'{pad}      write {w.name!r} at {where} pos={w.positions} {w.mechanism}({w.operand}) via {w.featurizer!r}')
                for r in tap.reads:
                    print(f'{pad}      read  {r.name!r} at {where} pos={r.positions} via {r.featurizer!r}')
    elif isinstance(step, Weights):
        print(f'{pad}{name}: Weights  names={list(step.names)}{tail}')


show(plan)

<root>: Plan
  featurizers: Featurizers
      rot: subspace k=8 d=16 seed=0 trained=True
  fit: Fit  10 epochs x 1 update  lr=0.001  objective=((1.0, 'ce'),)
      epochs[0][0]: Observe  metrics=['iia', 'ce']
          forward 'original' on 'counterfactual'  (2 rows x 11 tokens)
            read  'v_cf' at block_output[0] pos=((10,), (10,)) via 'rot'
          forward 'patched' on 'base'  (2 rows x 11 tokens)
            write 'patch' at block_output[0] pos=((10,), (10,)) swap(v_cf) via 'rot'
            read  'logits' at lm_head pos=((10,), (10,)) via 'identity'
      evaluation: Observe  metrics=['iia', 'ce']  saves=['held_out_iia.json']
          forward 'original' on 'counterfactual'  (2 rows x 9 tokens)
            read  'v_cf' at block_output[0] pos=((8,), (8,)) via 'rot'
          forward 'patched' on 'base'  (2 rows x 9 tokens)
            write 'patch' at block_output[0] pos=((8,), (8,)) swap(v_cf) via 'rot'
            read  'logits' at lm_head pos=((8,), (8,)) via 'identity'

Two things to notice.

**`d=16` was never authored.** The document says `k: 8`; the width came from asking the
engine how wide `block_output` is on this model.

**`pos=(10, 10)` and `pos=(8, 8)`.** The document says `pos: -1` everywhere. The fit's
rows are 11 tokens wide and the eval rows are 9, so the same `-1` resolved to different
absolute indices. Nothing inside the session decides that.

### A plan is data

Before it runs, the whole tree is strings, integers and tuples — no tensors, no model,
no tokenizer. That is what makes it shippable to a remote server.

In [7]:
print('pickles with plain pickle:', pickle.loads(pickle.dumps(plan)) == plan)
print('results before the run    :', {name: step.results for name, step in plan.steps.items()})

pickles with plain pickle: True
results before the run    : {'featurizers': {}, 'fit': {}, 'score': {}, 'weights': {}}


## 4. Running it

One `model.session(...)` for the whole request — the fit included. `remote=True` would
be the only change needed to run this on NDIF.

In [8]:
executed = engine.execute(plan)

for name, step in executed.steps.items():
    print(f'{name:12s} {sorted(step.results)}')
print(f"{'fit.evaluation':12s} {sorted(executed.step('fit', Fit).evaluation.results)}")

Loading weights:   0%|          | 0/21 [00:00<?, ?it/s]

featurizers  []
fit          ['train/eval', 'train/loss']
score        ['ce', 'iia']
weights      ['rot']
fit.evaluation ['ce', 'iia']


### The results are on the nodes that produced them

Both the fit's held-out pass and the scored pass produce a metric called `iia`. They
never collide, because they are different places.

In [9]:
fit = executed.step('fit', Fit)

print('scored pass  (trained-on rows) iia =', executed.step('score', Observe).results['iia'].tolist())
print('held-out pass (unseen rows)    iia =', fit.evaluation.results['iia'].tolist())
print()
print('loss per update :', fit.results['train/loss'].tolist())
print('eval per epoch  :', fit.results['train/eval'].squeeze(-1).tolist())

scored pass  (trained-on rows) iia = [0.01712493598461151, 0.002274729311466217]
held-out pass (unseen rows)    iia = [0.22498485445976257, -0.2277662456035614]

loss per update : [10.46630859375, 10.466291427612305, 10.46627426147461, 10.466259002685547]
eval per epoch  : [-0.0013846978545188904, -0.0013867318630218506, -0.0013887584209442139, -0.001390695571899414]


The fit stopped after 4 epochs of a 10-epoch budget: it early-stops on `iia`, `iia` fell
on every pass, and patience is 3. You can read every update where it happened:

In [10]:
for e, epoch in enumerate(fit.epochs[:4]):
    for update in epoch:
        print(f'epoch {e}:', {k: [round(x, 4) for x in v.tolist()] for k, v in update.results.items()})

epoch 0: {'iia': [0.0021, 0.017], 'ce': [10.4674, 10.4652]}
epoch 1: {'iia': [0.017, 0.0022], 'ce': [10.4652, 10.4674]}
epoch 2: {'iia': [0.0022, 0.0171], 'ce': [10.4674, 10.4651]}
epoch 3: {'iia': [0.0022, 0.0171], 'ce': [10.4674, 10.4651]}


### The rotation

In [11]:
from causalab_mini.ops import featurizer

weight = executed.result('rot')
basis = featurizer.cayley(weight)
print('weight       :', tuple(weight.shape))
print('QᵀQ == I     :', torch.allclose(basis.T @ basis, torch.eye(8), atol=1e-5))
print('moved off its start:', not torch.equal(weight, featurizer.start_weight(16, 8, 0)))

weight       : (16, 8)
QᵀQ == I     : True
moved off its start: True


## 5. What leaves the run

`Plan.write` walks the tree. A save writes its own step's result, so the held-out score
lands beside the scored one without either needing a qualified name.

In [12]:
import tempfile

out = pathlib.Path(tempfile.mkdtemp())
for path in executed.write(out):
    print(path.relative_to(out))

document.json
run.json
held_out_iia.json
iia.json
ce.json
rot.safetensors


In [13]:
rows = json.loads((out / 'held_out_iia.json').read_text())
rows

[{'example_id': '0',
  'metric': 'iia',
  'value': 0.22498485445976257,
  'eligible': True,
  'unit': 'logit',
  'estimand_version': 'logit_diff/v1',
  'produced_by': ''},
 {'example_id': '1',
  'metric': 'iia',
  'value': -0.2277662456035614,
  'eligible': True,
  'unit': 'logit',
  'estimand_version': 'logit_diff/v1',
  'produced_by': ''}]

## 6. The same experiment, in the protocol's format

[`documents/das_cpu_reduction.json`](../documents/das_cpu_reduction.json) is this run
written in causalab's own JSON. Different front end, same compiler underneath — so the
same rotation, to the bit.

In [14]:
protocol_raw = json.loads((REPO / 'documents' / 'das_cpu_reduction.json').read_text())
other = engine.execute(plan_module.build_request(protocol_raw, REPO / 'documents' / 'data', engine))

print('same rotation:', torch.equal(executed.result('rot'), other.result('rot')))
print('same scores  :', torch.equal(executed.step('score', Observe).results['iia'],
                                    other.step('observe', Observe).results['iia']))
print()
print('but the protocol version cannot save the held-out number:')
print('  its saves ->', [s.file_path for s in other.step('observe', Observe).saves])

same rotation: True
same scores  : True

but the protocol version cannot save the held-out number:
  its saves -> ['iia.json', 'ce.json']
